In [1]:
# =========================================
# MODEL 1 TRAINING - CATEGORY CLASSIFIER (OPTIMIZED)
# =========================================

import os
import json
import time
import random
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import ImageFolder
import torchvision.transforms as T
from sklearn.metrics import confusion_matrix, classification_report
import timm
from tqdm import tqdm

# -----------------------------------------
# CONFIG
# -----------------------------------------
SEED = 42
DATA_ROOT = "/kaggle/input/datasets/rupeshbhardwaj420/phase-1-2ndtime-updted/phase 1 II time updated"

OUT_DIR = Path("/kaggle/working/model1_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 260
BATCH_SIZE = 64
EPOCHS = 8
LR = 3e-4
WEIGHT_DECAY = 1e-4
MODEL_NAME = "efficientnet_b2"
VAL_RATIO = 0.2
NUM_WORKERS = 2 

# -----------------------------------------
# REPRODUCIBILITY
# -----------------------------------------
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

# -----------------------------------------
# CUSTOM DATASET WRAPPER (Handles Label Re-mapping)
# -----------------------------------------
class FilteredDataset(Dataset):
    def __init__(self, full_dataset, indices, label_map):
        self.full_dataset = full_dataset
        self.indices = indices
        self.label_map = label_map # Old index -> New index (0 to N-1)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        img, old_label = self.full_dataset[real_idx]
        return img, self.label_map[old_label]

    def __len__(self):
        return len(self.indices)

# -----------------------------------------
# TRANSFORMS
# -----------------------------------------
train_tf = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomRotation(10),
    T.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

val_tf = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.ToTensor(),
    T.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])

# -----------------------------------------
# DATASET LOADING & FILTERING
# -----------------------------------------
# Load base dataset twice (once for train transforms, once for val)
base_train = ImageFolder(DATA_ROOT, transform=train_tf)
base_val = ImageFolder(DATA_ROOT, transform=val_tf)

exclude_class = "Shapes"
old_class_to_idx = base_train.class_to_idx
exclude_idx = old_class_to_idx[exclude_class]

# Create new continuous label mapping
new_class_names = [cls for cls in base_train.classes if cls != exclude_class]
new_class_to_idx = {cls: i for i, cls in enumerate(new_class_names)}
label_remap = {old_class_to_idx[cls]: new_class_to_idx[cls] for cls in new_class_names}
num_classes = len(new_class_names)

# Instant filtering using .targets (no image loading)
all_labels = np.array(base_train.targets)
valid_indices = np.where(all_labels != exclude_idx)[0]
np.random.shuffle(valid_indices)

val_n = int(len(valid_indices) * VAL_RATIO)
val_idx = valid_indices[:val_n]
train_idx = valid_indices[val_n:]

train_ds = FilteredDataset(base_train, train_idx, label_remap)
val_ds = FilteredDataset(base_val, val_idx, label_remap)

print(f"Total valid images: {len(train_ds) + len(val_ds)}")
print(f"Classes remaining: {new_class_names}")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

# -----------------------------------------
# MODEL SETUP
# -----------------------------------------
model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

# -----------------------------------------
# EVALUATION & TRAINING
# -----------------------------------------
@torch.no_grad()
def evaluate():
    model.eval()
    y_true, y_pred = [], []
    running_val_loss = 0.0
    for x, y in val_loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        running_val_loss += loss.item() * x.size(0)
        y_true.extend(y.cpu().tolist())
        y_pred.extend(torch.argmax(logits, dim=1).cpu().tolist())
    
    acc = (np.array(y_true) == np.array(y_pred)).mean()
    return running_val_loss / len(val_ds), acc, y_true, y_pred

best_acc = 0.0
for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0
    for x, y in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}"):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(x), y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * x.size(0)

    val_loss, val_acc, y_true, y_pred = evaluate()
    print(f"Epoch {epoch}: Train Loss: {running_loss/len(train_ds):.4f} | Val Acc: {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), OUT_DIR / "model1_category_best.pt")

# -----------------------------------------
# SAVE OUTPUTS
# -----------------------------------------
(OUT_DIR / "class_to_idx.json").write_text(json.dumps(new_class_to_idx, indent=2))
print(f"\n✅ Done. Best Val Accuracy: {best_acc:.4f}")

Device: cuda
Total valid images: 31860
Classes remaining: ['Animals', 'Vechile', 'flags', 'flower', 'food', 'fruits and vegetables', 'monument', 'sports equipments', 'weather']


model.safetensors:   0%|          | 0.00/36.8M [00:00<?, ?B/s]

Epoch 1/8:  16%|█▌        | 63/399 [00:50<05:47,  1.03s/it]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Epoch 1/8:  31%|███       | 123/399 [01:37<03:43,  1.23it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Epoch 1/8: 100%|██████████| 399/399 [05:07<00:00,  1.30it/s]
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 1: Train Loss: 0.1577 | Val Acc: 0.9826


Epoch 2/8:   3%|▎         | 13/399 [00:09<04:18,  1.49it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Epoch 2/8:  48%|████▊     | 190/399 [02:03<01:39,  2.09it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Epoch 2/8: 100%|██████████| 399/399 [04:29<00:00,  1.48it/s]
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


...

Epoch 6: Train Loss: 0.0252 | Val Acc: 0.9859


Epoch 7/8:   0%|          | 0/399 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Epoch 7/8:  54%|█████▍    | 215/399 [02:07<01:44,  1.77it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Epoch 7/8: 100%|██████████| 399/399 [03:53<00:00,  1.71it/s]
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 7: Train Loss: 0.0164 | Val Acc: 0.9841


Epoch 8/8:   0%|          | 0/399 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Epoch 8/8:  21%|██        | 83/399 [00:51<03:30,  1.50it/s]/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(
Epoch 8/8: 100%|██████████| 399/399 [03:58<00:00,  1.68it/s]
/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 8: Train Loss: 0.0228 | Val Acc: 0.9849

✅ Done. Best Val Accuracy: 0.9859
